# Punjab InSAR + W3RA Bootstrap Notebook

This notebook checks GMTSAR time-series outputs for Punjab, loads W3RA hydrology data, and prepares SWIN-transformer-ready tensors.

Scope for now:
- Data QA and quick plotting
- Temporal alignment between InSAR and W3RA
- Physics-aware extension with a poroelastic Green's function layer (GRACE omitted)

In [ ]:
from pathlib import Path
import warnings
import math

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['image.cmap'] = 'RdBu_r'

In [ ]:
# Candidate paths (second path corrected from aoi_punjabCheck to aoi_punjab).
GMTSAR_CANDIDATES = [
    Path('/mnt/data/aoi_punjab_2'),
    Path('/mnt/data/aoi_punjab')
]
W3RA_DIR = Path('/mnt/data/w3ra_era5_10km_monthly_2003_2024')
W3RA_SYNTH_DIR = Path('/home/ubuntu/work/w3ra_synthetic_deformation')

def pick_gmtsar_dir(candidates):
    required = ['disp_all_ll.h5', 'coh_ll.h5', 'vel_ll.h5', 'aquisition_dates_ll.h5']
    for d in candidates:
        if d.exists() and all((d / f).exists() for f in required):
            return d
    raise FileNotFoundError('No GMTSAR directory contains all expected HDF5 files.')

GMTSAR_DIR = pick_gmtsar_dir(GMTSAR_CANDIDATES)
print('Using GMTSAR_DIR:', GMTSAR_DIR)
print('W3RA_DIR exists:', W3RA_DIR.exists())
print('W3RA synthetic dir exists:', W3RA_SYNTH_DIR.exists())

In [ ]:
def decode_dates(raw):
    out = []
    arr = np.array(raw).ravel()
    for item in arr:
        if isinstance(item, bytes):
            s = item.decode('utf-8', errors='ignore')
        elif isinstance(item, np.ndarray) and item.size == 1:
            v = item.item()
            if isinstance(v, bytes):
                s = v.decode('utf-8', errors='ignore')
            else:
                s = str(v)
        else:
            s = str(item)
        s = s.strip().replace('-', '').replace('/', '')
        out.append(s)
    return out

with h5py.File(GMTSAR_DIR / 'disp_all_ll.h5', 'r') as f_disp, \
     h5py.File(GMTSAR_DIR / 'coh_ll.h5', 'r') as f_coh, \
     h5py.File(GMTSAR_DIR / 'vel_ll.h5', 'r') as f_vel, \
     h5py.File(GMTSAR_DIR / 'aquisition_dates_ll.h5', 'r') as f_dates:

    lat = f_disp['lat'][:]
    lon = f_disp['lon'][:]
    z_shape = f_disp['z'].shape
    coh = f_coh['z'][:]
    vel = f_vel['z'][:]
    raw_dates = f_dates['acquisition_dates'][:]

acq_dates = decode_dates(raw_dates)
acq_dt = pd.to_datetime(acq_dates, format='%Y%m%d', errors='coerce')

print('lat shape:', lat.shape, 'lon shape:', lon.shape)
print('disp cube shape [T,Y,X]:', z_shape)
print('coherence shape:', coh.shape, 'velocity shape:', vel.shape)
print('number of acquisition dates:', len(acq_dates))
print('date range:', acq_dt.min(), 'to', acq_dt.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

im0 = ax[0].imshow(vel, origin='lower')
ax[0].set_title('GMTSAR Velocity (vel_ll.h5 /z)')
plt.colorbar(im0, ax=ax[0], fraction=0.045, pad=0.04)

im1 = ax[1].imshow(coh, origin='lower', cmap='viridis')
ax[1].set_title('GMTSAR Coherence (coh_ll.h5 /z)')
plt.colorbar(im1, ax=ax[1], fraction=0.045, pad=0.04)

for a in ax:
    a.set_xlabel('X pixel')
    a.set_ylabel('Y pixel')

plt.tight_layout()
plt.show()

In [ ]:
# Memory-safe access to large displacement cube.
with h5py.File(GMTSAR_DIR / 'disp_all_ll.h5', 'r') as f_disp:
    z_ds = f_disp['z']
    t_len, ny, nx = z_ds.shape
    sample_t = sorted({0, t_len // 2, t_len - 1})

    fig, ax = plt.subplots(1, len(sample_t), figsize=(15, 4))
    for i, ti in enumerate(sample_t):
        frame = z_ds[ti, :, :]
        q1, q99 = np.nanpercentile(frame, [1, 99])
        im = ax[i].imshow(frame, origin='lower', vmin=q1, vmax=q99)
        label = acq_dates[ti] if ti < len(acq_dates) else f't={ti}'
        ax[i].set_title(f'Displacement snapshot {label}')
        plt.colorbar(im, ax=ax[i], fraction=0.045, pad=0.04)
    plt.tight_layout()
    plt.show()

    # Plot a few pixel time series at stable/high-coherence locations.
    coh_mask = np.isfinite(coh)
    yy, xx = np.where(coh_mask)
    pick = np.linspace(0, len(yy) - 1, 4, dtype=int)

    plt.figure(figsize=(10, 5))
    for k in pick:
        y0, x0 = int(yy[k]), int(xx[k])
        ts = z_ds[:, y0, x0]
        plt.plot(acq_dt, ts, lw=1, label=f'pix({y0},{x0})')
    plt.title('Example displacement time series')
    plt.xlabel('Date')
    plt.ylabel('Displacement (raw GMTSAR units)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
w3ra_file = W3RA_DIR / 'Pinjab_W3RA_10km.mat'
mat = loadmat(w3ra_file)

S0 = np.asarray(mat['S0_pinjab'])
Sd = np.asarray(mat['Sd_pinjab'])
Sg = np.asarray(mat['Sg_pinjab'])
lat_w = np.asarray(mat['lat_pinjab']).ravel()
lon_w = np.asarray(mat['lon_pinjab']).ravel()

print('S0 shape:', S0.shape, 'Sd shape:', Sd.shape, 'Sg shape:', Sg.shape)
print('W3RA time steps:', Sg.shape[0], '(expected monthly 2003-2024 => 264)')
print('W3RA flattened pixels:', Sg.shape[1], '(sqrt ->', int(math.sqrt(Sg.shape[1])), ')')

grid_n = int(math.sqrt(Sg.shape[1]))
Sg_grid = Sg.reshape(Sg.shape[0], grid_n, grid_n)

plt.figure(figsize=(6, 5))
plt.imshow(Sg_grid[-1], origin='lower', cmap='viridis')
plt.title('W3RA Sg last timestep (reshaped 41x41)')
plt.colorbar(label='Sg')
plt.tight_layout()
plt.show()

w3ra_dates = pd.date_range('2003-01-01', periods=Sg.shape[0], freq='MS')
plt.figure(figsize=(10, 4))
plt.plot(w3ra_dates, np.nanmean(Sg, axis=1), label='mean Sg')
plt.plot(w3ra_dates, np.nanmean(Sd, axis=1), label='mean Sd')
plt.plot(w3ra_dates, np.nanmean(S0, axis=1), label='mean S0')
plt.title('W3RA basin-mean storage components (Punjab subset)')
plt.xlabel('Date')
plt.ylabel('Storage')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def nearest_downsample_2d(arr2d, out_h=41, out_w=41):
    yi = np.linspace(0, arr2d.shape[0] - 1, out_h).astype(int)
    xi = np.linspace(0, arr2d.shape[1] - 1, out_w).astype(int)
    return arr2d[np.ix_(yi, xi)]

# Build a monthly InSAR stack on 41x41 grid to align with W3RA resolution.
insar_month_df = pd.DataFrame({'date': acq_dt}).dropna()
insar_month_df['month'] = insar_month_df['date'].dt.to_period('M').dt.to_timestamp()
months = sorted(insar_month_df['month'].unique())
month_to_indices = {m: insar_month_df.index[insar_month_df['month'] == m].tolist() for m in months}

insar_monthly = []
insar_monthly_dates = []
with h5py.File(GMTSAR_DIR / 'disp_all_ll.h5', 'r') as f_disp:
    z_ds = f_disp['z']
    for m, idxs in month_to_indices.items():
        # Read only frames in this month, then average.
        block = z_ds[idxs, :, :]
        mean_frame = np.nanmean(block, axis=0)
        ds_frame = nearest_downsample_2d(mean_frame, 41, 41)
        insar_monthly.append(ds_frame)
        insar_monthly_dates.append(m)

insar_monthly = np.asarray(insar_monthly)  # [T_insar, 41, 41]
print('insar_monthly shape:', insar_monthly.shape)

common_months = sorted(set(insar_monthly_dates).intersection(set(w3ra_dates)))
print('common months with W3RA:', len(common_months))

if common_months:
    i_idx = [insar_monthly_dates.index(m) for m in common_months]
    w_idx = [np.where(w3ra_dates == m)[0][0] for m in common_months]
    insar_common = insar_monthly[i_idx]
    S0_common = S0[w_idx].reshape(len(common_months), 41, 41)
    Sd_common = Sd[w_idx].reshape(len(common_months), 41, 41)
    Sg_common = Sg[w_idx].reshape(len(common_months), 41, 41)

    print('aligned shapes -> InSAR:', insar_common.shape, 'S0:', S0_common.shape, 'Sd:', Sd_common.shape, 'Sg:', Sg_common.shape)

    plt.figure(figsize=(12, 4))
    plt.plot(common_months, np.nanmean(insar_common, axis=(1, 2)), label='mean InSAR (monthly)')
    plt.plot(common_months, np.nanmean(Sg_common, axis=(1, 2)), label='mean Sg')
    plt.title('Temporal overlap sanity check')
    plt.xlabel('Month')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# SWIN-ready input + poroelastic physics block (Algorithm 3 style, GRACE omitted).
import torch
import torch.nn as nn
import torch.nn.functional as F

def make_greens_kernel(kernel_size=9, alpha=1.5, eps=1e-6):
    c = kernel_size // 2
    yy, xx = np.mgrid[-c:c+1, -c:c+1]
    r = np.sqrt(xx**2 + yy**2) + eps
    # Simple isotropic decay kernel as a first poroelastic proxy.
    g = np.exp(-alpha * r) / r
    g[c, c] = g[c, c-1]  # avoid singular center
    g = g / g.sum()
    return torch.tensor(g, dtype=torch.float32)

class PoroelasticResponse(nn.Module):
    def __init__(self, kernel_size=9, alpha=1.5):
        super().__init__()
        k = make_greens_kernel(kernel_size, alpha).unsqueeze(0).unsqueeze(0)
        self.register_buffer('kernel', k)
        self.pad = kernel_size // 2

    def forward(self, x):
        # x: [B, 1, H, W] predicted storage-change field
        return F.conv2d(x, self.kernel, padding=self.pad)

if 'common_months' in globals() and len(common_months) > 0:
    # Build [B=1, C=4, T, H, W] = [InSAR, Sd, Sg, S0].
    x_input = np.stack([insar_common, Sd_common, Sg_common, S0_common], axis=0)
    x_input = np.transpose(x_input, (0, 1, 2, 3))  # [C,T,H,W]
    x_input = np.expand_dims(x_input, 0)            # [B,C,T,H,W]

    x_t = torch.tensor(x_input, dtype=torch.float32)
    print('SWIN input tensor shape [B,C,T,H,W]:', tuple(x_t.shape))

    poro = PoroelasticResponse(kernel_size=9, alpha=1.3)
    # Example physics transform on one timestep of one channel (Sg).
    sg_t0 = x_t[:, 2, 0, :, :].unsqueeze(1)
    u_elastic = poro(sg_t0)
    print('poroelastic response shape:', tuple(u_elastic.shape))

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(sg_t0[0, 0].numpy(), origin='lower', cmap='viridis')
    plt.title('Sg field (t0)')
    plt.colorbar(fraction=0.045, pad=0.04)

    plt.subplot(1, 2, 2)
    plt.imshow(u_elastic[0, 0].detach().numpy(), origin='lower', cmap='viridis')
    plt.title('Poroelastic response G * Sg')
    plt.colorbar(fraction=0.045, pad=0.04)

    plt.tight_layout()
    plt.show()
else:
    print('No common months found yet; run previous cells and verify date parsing.')

## Next Steps Toward Full Inversion

1. Replace isotropic proxy Green's kernel with calibrated poroelastic Green's function for layered media.
2. Add a physics-consistency loss term: `L_phys = ||u_pred - G * Sg_pred||`.
3. Train SWIN on synthetic W3RA-driven deformation first (`/home/ubuntu/work/w3ra_synthetic_deformation`), then fine-tune on real Punjab InSAR.
4. Add train/val/test splits by time windows, not random shuffling.
5. Keep GRACE branch disabled for now, consistent with your Algorithm 3 scope.

Optional Docker direction:
- Base from a CUDA-enabled PyTorch image.
- Add `h5py scipy xarray matplotlib pandas`.
- Mount `/mnt/data` and `/home/ubuntu/work` as volumes so notebook paths stay unchanged.